<a href="https://colab.research.google.com/github/WaelBeldi/checkpoint-web-scraping/blob/main/Checkpoint_Web_Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
# 1) Write a function to Get and parse html content from a Wikipedia page
import requests
from bs4 import BeautifulSoup

def get_html_content(url):
    """
    Fetches and parses HTML content from a Wikipedia page.

    Parameters:
        url (str): Wikipedia page URL
    Headers:
        User-Agent: Allow to access the page
    Returns:
        soup (BeautifulSoup): Parsed HTML content
    """

    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36'}

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        soup = BeautifulSoup(response.content, "html.parser")
        return soup
    else:
        print("Error fetching the page:", response.status_code)
        return None

soup = get_html_content("https://en.wikipedia.org/wiki/Python_(programming_language)")

In [54]:
# 2) Write a function to Extract article title
def get_article_title(soup):
    """
    Extracts the article title from a Wikipedia page.

    Parameters:
        soup (BeautifulSoup): Parsed HTML content

    Returns:
        title (str): Article title
    """
    if soup is None:
        return None

    title_tag = soup.find("h1", id="firstHeading")

    if title_tag:
        return title_tag.text.strip()
    else:
        return None

In [55]:
# 3) Write a function to Extract article text for each paragraph with their respective headings.
# Map those headings to their respective paragraphs in the dictionary.
def get_article_text_by_heading(soup):
    """
    Extracts article text from a Wikipedia page and maps
    each heading to its corresponding paragraphs.

    Parameters:
        soup (BeautifulSoup): Parsed HTML content

    Returns:
        content (dict): { heading : [paragraphs] }
    """
    if soup is None:
        return None

    content = {}
    current_heading = "Introduction"
    content[current_heading] = []

    body = soup.find("div", id="mw-content-text")

    for tag in body.find_all(["h2", "p"]):

        # New section
        if tag.name == "h2":
            current_heading = tag.text.strip()
            content[current_heading] = []

        # Paragraph
        elif tag.name == "p":
            if tag.text.strip():
                content[current_heading].append(tag.text.strip())

    return content

In [56]:
# 4) Write a function to collect every link that redirects to another Wikipedia page
def collect_wikipedia_links(soup):
    """
    Collects all internal links that redirect to other Wikipedia pages.

    Parameters:
        soup (BeautifulSoup): Parsed HTML content

    Returns:
        links (set): Unique internal Wikipedia links
    """
    links = set()

    for a_tag in soup.find_all("a", href=True):
        href = a_tag["href"]

        # Wikipedia internal links start with /wiki/
        if href.startswith("/wiki/") and ":" not in href:
            full_url = "https://en.wikipedia.org" + href
            links.add(full_url)

    return links

In [58]:
# 5) Wrap all the previous functions into a single function that takes as parameters a Wikipedia link
def extract_wikipedia_data(url):
    """
    Extracts structured data from a Wikipedia page.

    Parameters:
        url (str): Wikipedia page URL

    Returns:
        dict: {
            'title': article title,
            'content': text grouped by headings,
            'links': internal Wikipedia links
        }
    """
    soup = get_html_content(url)

    if soup is None:
        return None

    data = {
        "title": get_article_title(soup),
        "content": get_article_text_by_heading(soup),
        "links": collect_wikipedia_links(soup)
    }

    return data

In [59]:
# 6) Test the last function on a Wikipedia page of your choice
extract_wikipedia_data("https://en.wikipedia.org/wiki/Python_(programming_language)")

{'title': 'Python (programming language)',
 'content': {'Introduction': ['Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability with the use of significant indentation.[34] Python is dynamically type-checked and garbage-collected. It supports multiple programming paradigms, including structured (particularly procedural), object-oriented and functional programming.',
   'Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python\xa03.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5,[35] capabilities and keywords for typing were added to the language, allowing optional static typing.[36] As of 2025[update], the Python Software Foundation supports Python 3.10, 3.11, 3.12, 3.13, and 3.14, following the projects annual release cycle and five-year support policy. Earlier versions in the 3.x series hav